<!-- TARGET: public course-materials repo, sessions/m05/demo.ipynb (full version, after class). Copy to work/m05/ before editing. -->
# Module 5 · Live demo

**Python for Data Science · ESADE MIBA · Wed 11 Nov 2026**

The CFO asks for one number: *total revenue*. We have two candidates, 16.0 M and 20.3 M BRL.
By the end of today you can say which one is right, and prove it.

Two revenue numbers, used the same way in every notebook from now on:

- **Paid revenue** = `payment_value`, summed per order. What customers paid.
- **Item revenue** = `price + freight_value`, summed over item lines. What each item sold for.
  The only one that can be split by seller or category.

In [ ]:
from pathlib import Path

import pandas as pd

RAW = Path("../../data/raw")
CLEAN = Path("../../data/clean")

## 0. Raw → clean

`data/raw/` is read-only: what Olist gave us. `data/clean/` holds the same tables, **typed**:
one file per table, written by code, never by hand.

In [ ]:
DATE_COLUMNS = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
orders = pd.read_csv(RAW / "olist_orders_dataset.csv", parse_dates=DATE_COLUMNS)
CLEAN.mkdir(parents=True, exist_ok=True)
orders.to_parquet(CLEAN / "orders.parquet", index=False)

Why parquet and not CSV? Write the typed table to CSV, read it back, and compare the dtypes.

*Your turn (1 min):* predict the dtype of `order_purchase_timestamp` in each version. Then
read `orders.parquet` back into `orders_from_parquet`.

In [ ]:
orders.to_csv(CLEAN / "orders_roundtrip.csv", index=False)
orders_from_csv = pd.read_csv(CLEAN / "orders_roundtrip.csv")
orders_from_parquet = pd.read_parquet(CLEAN / "orders.parquet")

In [ ]:
orders_from_csv = pd.read_csv(CLEAN / "orders_roundtrip.csv")
orders_from_parquet = pd.read_parquet(CLEAN / "orders.parquet")
pd.DataFrame({"csv": orders_from_csv.dtypes, "parquet": orders_from_parquet.dtypes})

In [ ]:
(CLEAN / "orders_roundtrip.csv").unlink()

CSV is text: every type is lost on the way out. Parquet stores the types, and the file is
smaller. `pyarrow`, in your `pds` env since M01, is the engine that does it.

From here on, the clean layer is read with `pd.read_parquet`.

## 1. Grain: what is one row?

Before any `groupby` or `merge`, say what one row of each table **is**, then test it.

In [ ]:
orders = pd.read_parquet(CLEAN / "orders.parquet")
customers = pd.read_csv(RAW / "olist_customers_dataset.csv")
items = pd.read_csv(RAW / "olist_order_items_dataset.csv")
payments = pd.read_csv(RAW / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(RAW / "olist_order_reviews_dataset.csv")
sellers = pd.read_csv(RAW / "olist_sellers_dataset.csv")

### One row of `orders` is one order

In [ ]:
orders["order_id"].is_unique

### One row of `customers` is… a customer *per order*

In [ ]:
print(len(customers), customers["customer_id"].is_unique)
print(customers["customer_unique_id"].nunique())

`customer_id` is created for each order. The person is `customer_unique_id`: 99,441 rows,
96,096 people. "How many customers do we have?" has two answers. Only one of them is a person.

### One row of `items` is one item line of an order

In [ ]:
print(items["order_id"].is_unique)
print(items.duplicated(subset=["order_id", "order_item_id"]).any())

`order_id` alone is not unique: an order can have several items. The pair
`(order_id, order_item_id)` is. That pair is the **key** of the table.

*Your turn (2 min):* what is one row of `payments`? Find its key and prove it: assign to
`payments_key_is_unique` the result of checking that no row repeats the key.

In [ ]:
payments.head()
payments_key_is_unique = not payments.duplicated(subset=["order_id", "payment_sequential"]).any()

In [ ]:
assert not payments.duplicated(subset=["order_id", "payment_sequential"]).any()
print((payments["order_id"].value_counts() > 1).sum(), "orders paid in more than one payment")

### Reviews: one per order?

In [ ]:
print(reviews["order_id"].is_unique)
repeated_reviews = reviews[reviews["order_id"].duplicated(keep=False)]
print(repeated_reviews["order_id"].nunique(), "orders with more than one review")
repeated_reviews.sort_values("order_id").head(6)[["order_id", "review_score", "review_creation_date"]]

547 orders have two or more reviews. Any "average score per order" has to decide which one
counts. You make that decision in the lab.

**The rule:** an `assert` turns "I think one row is an order" into a check that fails loudly
the day it stops being true.

In [ ]:
assert orders["order_id"].is_unique
assert customers["customer_id"].is_unique
assert not items.duplicated(subset=["order_id", "order_item_id"]).any()

## 2. Split-apply-combine

**Split** the rows into groups, **apply** a function to each group, **combine** the results
into one table. In pandas: `groupby(...).agg(...)`.

### How do customers pay?

In [ ]:
payments.groupby("payment_type")["payment_value"].sum().sort_values(ascending=False)

One number per group. Usually the business wants several. **Named aggregations**: each output
column is `name=(column, function)`.

In [ ]:
by_payment_type = payments.groupby("payment_type").agg(
    paid_revenue=("payment_value", "sum"),
    payments=("payment_value", "size"),
    mean_payment=("payment_value", "mean"),
    mean_installments=("payment_installments", "mean"),
)
by_payment_type.sort_values("paid_revenue", ascending=False)

Credit cards carry three quarters of the revenue and are paid in 3.5 instalments on average.
The other methods always pay in one go.

### `size` vs `count`

In [ ]:
orders.groupby("order_status").agg(
    orders=("order_id", "size"),
    with_delivery_date=("order_delivered_customer_date", "count"),
)

`size` counts rows. `count` counts **non-missing** values. Same table, two different questions.
The 8 delivered orders without a date from M04 are visible here.

### The top sellers

In [ ]:
items = items.assign(item_revenue=items["price"] + items["freight_value"])
top_sellers = (
    items.groupby("seller_id")
    .agg(
        item_revenue=("item_revenue", "sum"),
        items_sold=("order_item_id", "size"),
        orders=("order_id", "nunique"),
    )
    .sort_values("item_revenue", ascending=False)
)
top_sellers.head(10)

`items_sold` and `orders` differ: a seller can ship several items in one order. `nunique`
counts orders, not lines.

*Your turn (3 min):* among sellers with **at least 100 orders**, which ten have the highest
mean item revenue per item sold? Store the ten rows in `premium_sellers`.

In [ ]:
busy_sellers = top_sellers[top_sellers["orders"] >= 100]
premium_sellers = (
    busy_sellers.assign(revenue_per_item=busy_sellers["item_revenue"] / busy_sellers["items_sold"])
    .sort_values("revenue_per_item", ascending=False)
    .head(10)
)
premium_sellers

### Changing the grain on purpose

Payments are at payment grain. Most questions are about orders. Group by `order_id` and you
get **one row per order**: a new table at a new grain.

In [ ]:
paid_by_order = payments.groupby("order_id").agg(paid_revenue=("payment_value", "sum")).reset_index()
assert paid_by_order["order_id"].is_unique
print(len(payments), "payments →", len(paid_by_order), "orders")
print(f"paid revenue: {paid_by_order['paid_revenue'].sum():,.2f} BRL")

## 3. Joins

### Revenue by customer state

`customer_state` lives in `customers`, the revenue in `payments`. A question that needs two
tables needs a `merge`.

In [ ]:
orders_with_state = orders.merge(
    customers[["customer_id", "customer_state"]], on="customer_id", how="left", validate="one_to_one"
)
assert len(orders_with_state) == len(orders)

`validate="one_to_one"` writes our grain assumption into the join. If either side had
repeated keys, pandas would stop with an error instead of silently multiplying rows.

Now the paid revenue, already at order grain. `indicator=True` adds a `_merge` column that
says where each row came from.

In [ ]:
orders_with_paid = orders_with_state.merge(
    paid_by_order, on="order_id", how="left", validate="one_to_one", indicator=True
)
orders_with_paid["_merge"].value_counts()

In [ ]:
assert len(orders_with_paid) == len(orders)
revenue_by_state = (
    orders_with_paid.groupby("customer_state")
    .agg(orders=("order_id", "size"), paid_revenue=("paid_revenue", "sum"))
    .sort_values("paid_revenue", ascending=False)
)
assert abs(revenue_by_state["paid_revenue"].sum() - payments["payment_value"].sum()) < 0.01
revenue_by_state.head()

São Paulo alone: 42% of the orders, 37% of the revenue. And the last `assert` proves the
states add up to the payments table: nothing lost, nothing added.

### `inner` vs `left`: who disappears?

*Your turn (1 min):* predict how many rows `orders.merge(items, on="order_id")` returns
(inner join, the default). There are 99,441 orders and 112,650 item lines. Store the actual
number in `n_inner`.

In [ ]:
n_inner = len(orders.merge(items, on="order_id"))
n_inner

In [ ]:
orders_and_items = orders.merge(items, on="order_id", how="left", indicator=True)
print(len(orders.merge(items, on="order_id")), "rows with inner,", len(orders_and_items), "with left")
orders_and_items["_merge"].value_counts()

In [ ]:
orders_without_items = orders_and_items[orders_and_items["_merge"] == "left_only"]
orders_without_items["order_status"].value_counts()

775 orders have no items: mostly unavailable or cancelled. An inner join drops them **without a
word**. Sometimes that's what you want; decide it, don't let the default decide for you.

### The 20.3 M BRL

Sales wants paid revenue *by seller state*. The seller lives in `items`, the money in
`payments`. The obvious join:

In [ ]:
items_and_payments = items.merge(payments, on="order_id")
print(len(items), "items,", len(payments), "payments →", len(items_and_payments), "rows")
print(f"'revenue': {items_and_payments['payment_value'].sum():,.2f} BRL")

**20.3 M.** An order with 2 items and 2 payments becomes 2 × 2 = 4 rows, and each payment
is counted twice. This is a **fan-out**: a many-to-many join on `order_id`. `validate` catches it:

In [ ]:
try:
    items.merge(payments, on="order_id", validate="many_to_one")
except pd.errors.MergeError as error:
    print("MergeError:", error)

(The `try`/`except` lets the notebook carry on so we can read the error. In your own work, let
it fail.)

Fix attempt: aggregate payments to order grain first, then join.

In [ ]:
items_and_paid = items.merge(paid_by_order, on="order_id", validate="many_to_one")
print(f"'revenue': {items_and_paid['paid_revenue'].sum():,.2f} BRL")

The join is valid now, yet the total is **still 20.3 M**. The order total sits on every item
line, so summing it over item lines counts each order once per item.

**The rule:** sum a number only at its own grain. Paid revenue is an order number: sum it over
orders. To split revenue by seller, use the number that lives on the item line, **item revenue**.

*Your turn (3 min):* join `items` to `sellers` (validate the grain!) and compute item revenue
by `seller_state` into `revenue_by_seller_state`, sorted from highest to lowest.

In [ ]:
items_with_seller = items.merge(
    sellers[["seller_id", "seller_state"]], on="seller_id", how="left", validate="many_to_one"
)
revenue_by_seller_state = (
    items_with_seller.groupby("seller_state")
    .agg(item_revenue=("item_revenue", "sum"))
    .sort_values("item_revenue", ascending=False)
)
revenue_by_seller_state.head()

### Prove it

In [ ]:
items_with_seller = items.merge(
    sellers[["seller_id", "seller_state"]], on="seller_id", how="left", validate="many_to_one"
)
revenue_by_seller_state = items_with_seller.groupby("seller_state").agg(item_revenue=("item_revenue", "sum"))

assert len(items_with_seller) == len(items)
assert abs(revenue_by_seller_state["item_revenue"].sum() - items["item_revenue"].sum()) < 0.01
print(f"item revenue: {items['item_revenue'].sum():,.2f} BRL")
print(f"paid revenue: {payments['payment_value'].sum():,.2f} BRL")

15.84 M by items, 16.01 M paid. The gap is about 1%: vouchers and instalment interest that sit
in payments but not on the item line. Both numbers are right; they answer different questions.
The 20.3 M answers none.

**For the CFO: 16.0 M BRL**, and the three `assert`s that prove it.

## Today in one table

| Step | Check |
|---|---|
| Write the clean layer | read it back: types survived |
| State the grain | `is_unique`, `duplicated(subset=…)` |
| Change the grain | `groupby(key).agg(name=(col, func))`, then assert the key is unique |
| Join | `validate=`, `indicator=True`, assert the row count |
| Sum | only at the number's own grain; assert totals match the source |